# Generate admission-level predictions

Loads fitted models and creates one common prediction table for downstream evaluation, temporal analysis, subgroup analysis, and explainability.

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from pathlib import Path
import joblib
import pandas as pd
import yaml
from icare_risk.data.audit import load_admission_file
from icare_risk.models.prediction import generate_prediction_table

from icare_risk.utils.paths import ANALYSIS_YAML_PATH
with ANALYSIS_YAML_PATH.open() as f:
    config = yaml.safe_load(f)

df = load_admission_file(config['outputs']['processed_dataset'])
model_directory = Path(config['outputs']['model_directory'])
fitted_models = {}
feature_map = {}
for model_cfg in config['models']:
    name = model_cfg['name']
    bundle = joblib.load(model_directory / f'{name}.joblib')
    fitted_models[name] = bundle['model']
    feature_map[name] = bundle['features']

prediction_table = generate_prediction_table(
    df, fitted_models, feature_map,
    id_columns=[config['columns']['admission_id'], config['columns']['patient_id']],
    target_column=config['columns']['target'],
    sort_by_model=None,  # e.g. 'xgb_all_scores' to sort highest risk first
)
display(prediction_table.head())

,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,GENDER_DESC,UPDATE_DT_TM,esbl_label_test,split,lr_kim_components_prob,lr_all_score_values_prob,lr_clinical_baseline_prob,xgb_kim_components_prob,xgb_all_score_values_prob,xgb_clinical_baseline_prob
0,10001,7867287,2998828,3749211,2023-02-12,00:52:00,64,2023-02-26,140,oral surgery,...,Male,2023-01-05,0,test,0.201013,0.203749,0.205190,0.200879,0.206126,0.176195
1,10002,6551735,8909338,9353593,2022-11-16,23:43:00,72,2022-12-01,320,cardiology,...,Unknown,2023-07-03,0,validation,0.201013,0.207353,0.206913,0.200879,0.209951,0.177484
2,10003,4441228,1307783,3518105,2022-05-12,20:10:00,29,2022-05-27,320,cardiology,...,Unknown,2023-10-23,1,train,0.201013,0.196683,0.197776,0.200879,0.193020,0.198216
3,10004,5593202,5695692,6593442,2023-10-11,00:52:00,19,2023-10-30,300,general medicine,...,Indeterminate,2023-10-10,0,test,0.201013,0.218444,0.195695,0.200879,0.430900,0.207285
4,10005,6532182,7455884,6203572,2023-08-14,05:42:00,50,2023-08-18,300,general medicine,...,Indeterminate,2023-01-06,0,test,0.201013,0.200193,0.202200,0.200879,0.196593,0.220026


In [6]:
prediction_directory = Path(config["outputs"].get("prediction_directory"))
output_path = prediction_directory / 'admission_predictions.parquet'
output_path.parent.mkdir(parents=True, exist_ok=True)
prediction_table.to_parquet(output_path, index=False)
print(f'Saved {len(prediction_table):,} admission predictions to {output_path.resolve()}')

Saved 10,000 admission predictions to results/predictions/admission_predictions.parquet
